# 🔬 Embedding Frobenius Norm — Shrinkage vs. Orthogonality Diagnostic

## Motivação (Devil's Advocate Hypothesis)

O orthogonal loss penaliza `mean((z_drv · z_cons)²)`. Mas existe um jeito "trapaça"
de satisfazer essa loss: **encolher a magnitude dos embeddings** até o dot product
ir pra zero, sem aprender estrutura ortogonal nenhuma.

$$
\text{dot}(a, b) = \|a\| \cdot \|b\| \cdot \cos(\theta)
$$

Se $\|a\|$ ou $\|b\|$ colapsam pra ~0, o dot product zera **independentemente do ângulo**.
A "ortogonalidade" vira shrinkage.

### O que este notebook verifica

| Métrica | O que revela |
|---------|-------------|
| **Frobenius norm** de E_drivers e E_constructors | Magnitude total dos embeddings |
| **L2 norm por embedding** (média, std, min, max) | Distribuição das magnitudes |
| **Cosine similarity média** entre pares (driver, constructor) | Ortogonalidade real (independente de norma) |
| **Norma dos pesos do classifier** | Se embeddings encolhem, classifier compensa? |
| **Evolução temporal** das métricas durante treinamento | Colapso gradual ou súbito? |

### Diagnóstico

- Se **normas comparáveis** entre λ=0, λ=0.1, λ=1.0 → ✅ Devil's Advocate refutado
- Se **normas caem drasticamente** com λ maior → ❌ Shrinkage confirmado
- Se **normas caem pouco mas cosine cai muito** → ✅ Ortogonalidade genuína

In [ ]:
import sys, os, warnings, json
sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from models.pipeline_fusion import F1OrthogonalPipeline
from relbench.datasets import get_dataset
from train import filter_db_by_years, build_graph, _build_instances
import config as cfg

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
LATENT_DIM = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Todas as 3 variantes do ablation study
MODEL_SPECS = [
    {"name": "model_no_orthogonal", "label": "λ = 0 (No Ortho)", "lambda": 0.0},
    {"name": "model_ablation_l01", "label": "λ = 0.1 (Low)", "lambda": 0.1},
    {"name": "model_orthogonal", "label": "λ = 1.0 (High)", "lambda": 1.0},
]

for spec in MODEL_SPECS:
    path = f"../output/models/{spec['name']}.pth"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}. Train first.")

print(f"Device: {DEVICE}")
print(f"Models to analyze: {[s['name'] for s in MODEL_SPECS]}")

In [ ]:
# ---------------------------------------------------------------------------
# 1. Load DB and build graph
# ---------------------------------------------------------------------------
print("Loading RelBench dataset & building graph...")
dataset = get_dataset(cfg.RELBENCH_DATASET, download=True)
db = dataset.get_db(upto_test_timestamp=False)
db = filter_db_by_years(db, 2000, 2023)

graph_data, node_to_col_names_dict, node_to_col_stats = build_graph(db)
graph_cpu = graph_data
graph_data = graph_data.to(DEVICE)

num_nodes_dict = {nt: graph_data[nt].num_nodes for nt in graph_data.node_types}

print(f"Node types: {graph_data.node_types}")
print(f"  drivers: {graph_data['drivers'].num_nodes}")
print(f"  constructors: {graph_data['constructors'].num_nodes}")
print(f"Edge types: {len(graph_data.edge_types)}")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Load models (with compatibility handling)
# ---------------------------------------------------------------------------
def load_model_with_compat(name, num_nodes_dict, node_to_col_names_dict, node_to_col_stats):
    """Carrega modelo salvo com tratamento de schema drift."""
    model = F1OrthogonalPipeline(
        num_nodes_dict=num_nodes_dict,
        node_to_col_names_dict=node_to_col_names_dict,
        node_to_col_stats=node_to_col_stats,
        latent_dim=LATENT_DIM,
    ).to(DEVICE)

    # Materializar lazy parameters
    if model.encoder is not None:
        with torch.no_grad():
            x_init = model.encoder(graph_cpu.tf_dict)
            _ = model.graph_encoder(x_init, graph_data.edge_index_dict)

    path = f"../output/models/{name}.pth"
    ckpt = torch.load(path, map_location=DEVICE)

    model_state = model.state_dict()
    compatible_state = {}
    skipped_shapes = []

    for k, v in ckpt.items():
        if k not in model_state:
            continue
        if model_state[k].shape != v.shape:
            skipped_shapes.append((k, tuple(v.shape), tuple(model_state[k].shape)))
            continue
        compatible_state[k] = v

    missing, unexpected = model.load_state_dict(compatible_state, strict=False)
    
    if skipped_shapes:
        print(f"  ⚠ {name}: {len(skipped_shapes)} tensors with schema mismatch (encoder drift)")
    
    model.eval()
    return model

models = {}
for spec in MODEL_SPECS:
    print(f"Loading {spec['label']}...")
    models[spec["name"]] = load_model_with_compat(
        spec["name"], num_nodes_dict, node_to_col_names_dict, node_to_col_stats
    )

print("All models loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# 3. Extract embeddings from all model variants
# ---------------------------------------------------------------------------
embeddings = {}  # model_name -> {"drivers": Tensor, "constructors": Tensor}

for spec in MODEL_SPECS:
    name = spec["name"]
    label = spec["label"]
    model = models[name]

    with torch.no_grad():
        x_dict = model.encoder(graph_cpu.tf_dict) if model.encoder is not None else graph_cpu.x_dict
        out_dict = model.graph_encoder(x_dict, graph_data.edge_index_dict)

    drv = out_dict["drivers"]  # (N_drivers, 32)
    cons = out_dict["constructors"]  # (N_constructors, 32)

    embeddings[name] = {"drivers": drv, "constructors": cons}
    print(f"{label}: drivers {tuple(drv.shape)}, constructors {tuple(cons.shape)}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Core diagnostic: Frobenius norm + L2 per-sample norms
# ---------------------------------------------------------------------------
def compute_norm_stats(embeddings_dict):
    """
    Para cada variante, computa:
    - Frobenius norm (magnitude total da matriz de embeddings)
    - Estatisticas das normas L2 por embedding (media, std, min, max)
    - Frobenius normalizada por sqrt(N) (norma media por embedding)
    """
    stats = {}
    for model_name, embs in embeddings_dict.items():
        stats[model_name] = {}
        for ent_type in ["drivers", "constructors"]:
            E = embs[ent_type]
            frob = torch.norm(E, p='fro').item()
            per_sample_norms = torch.norm(E, p=2, dim=1)
            n = E.shape[0]
            
            stats[model_name][ent_type] = {
                "frobenius": frob,
                "frobenius_per_sample": frob / np.sqrt(n),
                "n_samples": n,
                "mean_l2": per_sample_norms.mean().item(),
                "std_l2": per_sample_norms.std().item(),
                "min_l2": per_sample_norms.min().item(),
                "max_l2": per_sample_norms.max().item(),
                "cv_l2": (per_sample_norms.std() / (per_sample_norms.mean() + 1e-8)).item(),
            }
    return stats

norm_stats = compute_norm_stats(embeddings)

# Build comparison table
rows = []
for spec in MODEL_SPECS:
    name = spec["name"]
    for ent_type in ["drivers", "constructors"]:
        s = norm_stats[name][ent_type]
        rows.append({
            "model": spec["label"],
            "lambda": spec["lambda"],
            "entity": ent_type,
            "frobenius": s["frobenius"],
            "frob_per_sample": s["frobenius_per_sample"],
            "mean_l2": s["mean_l2"],
            "std_l2": s["std_l2"],
            "min_l2": s["min_l2"],
            "max_l2": s["max_l2"],
            "cv_l2": s["cv_l2"],
        })

norm_df = pd.DataFrame(rows)

print("=" * 90)
print("EMBEDDING NORM COMPARISON — Shrinkage Diagnostic")
print("=" * 90)
display(norm_df.round(4))

In [ ]:
# ---------------------------------------------------------------------------
# 5. Shrinkage ratio: how much norms shrink as λ increases
# ---------------------------------------------------------------------------
print("=" * 90)
print("SHRINKAGE RATIOS (relative to λ=0 baseline)")
print("=" * 90)

baseline = "model_no_orthogonal"
for spec in MODEL_SPECS:
    name = spec["name"]
    if name == baseline:
        continue
    for ent_type in ["drivers", "constructors"]:
        base_frob = norm_stats[baseline][ent_type]["frobenius"]
        curr_frob = norm_stats[name][ent_type]["frobenius"]
        ratio = curr_frob / base_frob
        
        base_mean = norm_stats[baseline][ent_type]["mean_l2"]
        curr_mean = norm_stats[name][ent_type]["mean_l2"]
        mean_ratio = curr_mean / base_mean
        
        print(f"{spec['label']:25s} {ent_type:15s}  "
              f"Frob: {curr_frob:.2f} / {base_frob:.2f} = {ratio:.4f}  |  "
              f"Mean L2: {curr_mean:.2f} / {base_mean:.2f} = {mean_ratio:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# 6. Cosine similarity: orthogonality independent of norm
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print("COSINE SIMILARITY — Orthogonality independent of magnitude")
print("=" * 90)

cosine_rows = []
for spec in MODEL_SPECS:
    name = spec["name"]
    drv = embeddings[name]["drivers"]
    cons = embeddings[name]["constructors"]
    
    # Normalize to unit vectors
    drv_n = torch.nn.functional.normalize(drv, p=2, dim=1)
    cons_n = torch.nn.functional.normalize(cons, p=2, dim=1)
    
    # Cosine matrix: all driver-constructor pairs
    cos_mat = torch.mm(drv_n, cons_n.T)  # (N_drv, N_cons)
    cos_abs = torch.abs(cos_mat)
    
    # Cosine of paired (driver, constructor) from same result — NOT available
    # without instance-level pairing, so we use global all-pairs cosine.
    # This is still informative: if orthogonal model truly decorrelates,
    # the global cosine distribution should be more concentrated near zero.
    
    cosine_rows.append({
        "model": spec["label"],
        "lambda": spec["lambda"],
        "mean_cos": cos_mat.mean().item(),
        "mean_abs_cos": cos_abs.mean().item(),
        "std_cos": cos_mat.std().item(),
        "frac_near_zero": (cos_abs < 0.1).float().mean().item(),
        "frac_high_cos": (cos_abs > 0.5).float().mean().item(),
    })

cosine_df = pd.DataFrame(cosine_rows)
display(cosine_df.round(4))

# Key diagnostic:
# - If mean_cos drops with λ but mean_abs_cos stays HIGH:
#   → symmetric (positive & negative cancel) but NOT orthogonal
# - If mean_abs_cos drops with λ:
#   → genuine orthogonal structure
# - If mean_abs_cos is similar across λ:
#   → no structural orthogonality learned

In [ ]:
# ---------------------------------------------------------------------------
# 7. Classifier weight norms: compensation check
# ---------------------------------------------------------------------------
print("\n" + "=" * 90)
print("CLASSIFIER WEIGHT NORMS — Does the classifier compensate for small embeddings?")
print("=" * 90)

classifier_rows = []
for spec in MODEL_SPECS:
    name = spec["name"]
    model = models[name]
    
    # classifier is nn.Sequential: Linear(64, 32), ReLU, Linear(32, 1)
    w1 = model.classifier[0].weight  # (32, 64)
    w2 = model.classifier[2].weight  # (1, 32)
    
    frob_w1 = torch.norm(w1, p='fro').item()
    frob_w2 = torch.norm(w2, p='fro').item()
    
    # aux heads
    aux_drv_w = torch.norm(model.aux_piloto.weight, p='fro').item()
    aux_cons_w = torch.norm(model.aux_equipe.weight, p='fro').item()
    
    classifier_rows.append({
        "model": spec["label"],
        "lambda": spec["lambda"],
        "classifier_w1_frob": frob_w1,
        "classifier_w2_frob": frob_w2,
        "aux_driver_frob": aux_drv_w,
        "aux_constructor_frob": aux_cons_w,
    })

classifier_df = pd.DataFrame(classifier_rows)
display(classifier_df.round(4))

print("\nInterpretation:")
print("- If classifier weights are LARGER for high-λ models → classifier compensates for small embeddings")
print("- If classifier weights are similar → embeddings carry similar information content regardless of norm")

In [ ]:
# ---------------------------------------------------------------------------
# 8. Combined diagnostic table
# ---------------------------------------------------------------------------
print("\n" + "=" * 100)
print("FINAL DIAGNOSTIC SUMMARY")
print("=" * 100)

summary_rows = []
for spec in MODEL_SPECS:
    name = spec["name"]
    s_drv = norm_stats[name]["drivers"]
    s_cons = norm_stats[name]["constructors"]
    c = cosine_df[cosine_df["model"] == spec["label"]].iloc[0]
    clf = classifier_df[classifier_df["model"] == spec["label"]].iloc[0]
    
    summary_rows.append({
        "Model": spec["label"],
        "λ": spec["lambda"],
        "||E_drv||_F": f"{s_drv['frobenius']:.1f}",
        "||E_cons||_F": f"{s_cons['frobenius']:.1f}",
        "mean_L2_drv": f"{s_drv['mean_l2']:.3f}",
        "mean_L2_cons": f"{s_cons['mean_l2']:.3f}",
        "CV_drv%": f"{s_drv['cv_l2']*100:.1f}",
        "CV_cons%": f"{s_cons['cv_l2']*100:.1f}",
        "mean|cos|": f"{c['mean_abs_cos']:.4f}",
        "%near_zero": f"{c['frac_near_zero']*100:.1f}",
        "%high_cos": f"{c['frac_high_cos']*100:.1f}",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("\n📋 Reading guide:")
print("  CV% (coefficient of variation): low → all embeddings have similar magnitude")
print("  mean|cos|: average absolute cosine driver-constructor — lower = more orthogonal")
print("  %near_zero: fraction of pairs with |cos| < 0.1 — higher = more orthogonal")
print("  %high_cos: fraction of pairs with |cos| > 0.5 — lower = more orthogonal")

In [ ]:
# ---------------------------------------------------------------------------
# 9. Visualizations
# ---------------------------------------------------------------------------
import os
os.makedirs("./output", exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]  # green (λ=0), orange (λ=0.1), red (λ=1.0)

# --- 9a. Frobenius norm bar chart ---
x = np.arange(len(MODEL_SPECS))
width = 0.35
for i, ent_type in enumerate(["drivers", "constructors"]):
    frobs = [norm_stats[s["name"]][ent_type]["frobenius"] for s in MODEL_SPECS]
    offset = (i - 0.5) * width
    axes[0, 0].bar(x + offset, frobs, width, label=ent_type, alpha=0.85)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels([s["label"] for s in MODEL_SPECS], fontsize=9)
axes[0, 0].set_title("Frobenius Norm of Embedding Matrix", fontsize=13, weight="bold")
axes[0, 0].set_ylabel("||E||_F")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.2, axis='y')

# --- 9b. Mean L2 norm per embedding ---
for i, ent_type in enumerate(["drivers", "constructors"]):
    means = [norm_stats[s["name"]][ent_type]["mean_l2"] for s in MODEL_SPECS]
    stds = [norm_stats[s["name"]][ent_type]["std_l2"] for s in MODEL_SPECS]
    offset = (i - 0.5) * width
    axes[0, 1].bar(x + offset, means, width, yerr=stds, label=ent_type, alpha=0.85, capsize=4)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([s["label"] for s in MODEL_SPECS], fontsize=9)
axes[0, 1].set_title("Mean L2 Norm per Embedding (±1σ)", fontsize=13, weight="bold")
axes[0, 1].set_ylabel("Mean ||v_i||_2")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.2, axis='y')

# --- 9c. Mean absolute cosine ---
mean_abs_cos_vals = [cosine_df[cosine_df["model"] == s["label"]]["mean_abs_cos"].values[0] for s in MODEL_SPECS]
axes[0, 2].bar(x, mean_abs_cos_vals, color=colors, alpha=0.85, edgecolor="black", linewidth=0.5)
axes[0, 2].set_xticks(x)
axes[0, 2].set_xticklabels([s["label"] for s in MODEL_SPECS], fontsize=9)
axes[0, 2].set_title("Mean |cos(driver, constructor)|", fontsize=13, weight="bold")
axes[0, 2].set_ylabel("Mean |cos|")
axes[0, 2].grid(alpha=0.2, axis='y')

# --- 9d. L2 norm histogram (drivers) ---
for spec, color in zip(MODEL_SPECS, colors):
    name = spec["name"]
    norms = torch.norm(embeddings[name]["drivers"], p=2, dim=1).cpu().numpy()
    axes[1, 0].hist(norms, bins=40, alpha=0.5, color=color, label=spec["label"], density=True)
axes[1, 0].set_title("Driver Embedding L2 Norm Distribution", fontsize=13, weight="bold")
axes[1, 0].set_xlabel("||v||_2")
axes[1, 0].set_ylabel("Density")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.2)

# --- 9e. L2 norm histogram (constructors) ---
for spec, color in zip(MODEL_SPECS, colors):
    name = spec["name"]
    norms = torch.norm(embeddings[name]["constructors"], p=2, dim=1).cpu().numpy()
    axes[1, 1].hist(norms, bins=40, alpha=0.5, color=color, label=spec["label"], density=True)
axes[1, 1].set_title("Constructor Embedding L2 Norm Distribution", fontsize=13, weight="bold")
axes[1, 1].set_xlabel("||v||_2")
axes[1, 1].set_ylabel("Density")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.2)

# --- 9f. Cosine distribution comparison ---
for spec, color in zip(MODEL_SPECS, colors):
    name = spec["name"]
    drv_n = torch.nn.functional.normalize(embeddings[name]["drivers"], p=2, dim=1)
    cons_n = torch.nn.functional.normalize(embeddings[name]["constructors"], p=2, dim=1)
    cos_vals = torch.mm(drv_n, cons_n.T).cpu().numpy().ravel()
    # Subsample for performance
    if len(cos_vals) > 50000:
        cos_vals = np.random.RandomState(42).choice(cos_vals, size=50000, replace=False)
    axes[1, 2].hist(cos_vals, bins=80, alpha=0.4, color=color, label=spec["label"], density=True)

axes[1, 2].axvline(x=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[1, 2].set_title("Driver-Constructor Cosine Distribution", fontsize=13, weight="bold")
axes[1, 2].set_xlabel("cos(driver, constructor)")
axes[1, 2].set_ylabel("Density")
axes[1, 2].legend(fontsize=8)
axes[1, 2].grid(alpha=0.2)

fig.suptitle("Embedding Frobenius Norm Analysis — Shrinkage vs. Genuine Orthogonality",
             fontsize=16, weight="bold", y=1.01)
plt.tight_layout()
plt.savefig("./output/embedding_frobenius_diagnostic.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 10. SHAP-style: per-dimension variance analysis
# ---------------------------------------------------------------------------
# If orthogonalization forces dimensions to be used more efficiently,
# we should see different patterns in per-dimension variance.

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, ent_type, title in zip(axes, ["drivers", "constructors"], ["Driver", "Constructor"]):
    for spec, color, marker in zip(MODEL_SPECS, colors, ["o", "s", "^"]):
        name = spec["name"]
        E = embeddings[name][ent_type]
        dim_vars = E.var(dim=0).cpu().numpy()  # variance per dimension
        dim_vars_sorted = np.sort(dim_vars)[::-1]
        ax.plot(range(1, len(dim_vars_sorted) + 1), dim_vars_sorted,
                color=color, marker=marker, markersize=4, linewidth=1.5,
                label=spec["label"], markevery=3)
    ax.set_title(f"{title} Dimension Variance (sorted)", fontsize=13, weight="bold")
    ax.set_xlabel("Dimension rank")
    ax.set_ylabel("Variance")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle("Per-Dimension Embedding Variance", fontsize=15, weight="bold")
plt.tight_layout()
plt.savefig("./output/embedding_frobenius_dimvar.png", dpi=220, bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("- Faster variance decay → fewer dimensions used → potential collapse")
print("- Similar variance curves across λ → similar dimensional utilization → ok")

In [ ]:
# ---------------------------------------------------------------------------
# 11. Training history: did norms evolve differently during training?
# ---------------------------------------------------------------------------
with open("../output/models/training_results.json") as f:
    results = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metric_key in enumerate(["val_orth", "val_auroc", "val_loss"]):
    ax = axes[i]
    for r, color in zip(results, colors):
        history = r["history"]
        name = r["model_name"]
        label_map = {
            "model_no_orthogonal": "λ=0",
            "model_ablation_l01": "λ=0.1",
            "model_orthogonal": "λ=1.0",
        }
        label = label_map.get(name, name)
        vals = history.get(metric_key, [])
        if vals:
            ax.plot(range(1, len(vals) + 1), vals, color=color, linewidth=2,
                    marker="o", markersize=4, label=label)
    
    titles = {
        "val_orth": "Validation Orthogonal Loss",
        "val_auroc": "Validation AUROC",
        "val_loss": "Validation Total Loss",
    }
    ax.set_title(titles.get(metric_key, metric_key), fontsize=13, weight="bold")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle("Training History Comparison", fontsize=15, weight="bold")
plt.tight_layout()
plt.savefig("./output/embedding_frobenius_history.png", dpi=220, bbox_inches="tight")
plt.show()

print("\nNote on orth loss for λ=0:")
print("  The no-orthogonal model still computes orth loss (it's just not backpropped through).")
print("  High orth loss at λ=0 means embeddings ARE correlated; the loss term IS measuring something.")
print("  The steep drop for λ=0.1 and λ=1.0 shows the constraint is being actively enforced.")

In [ ]:
# ---------------------------------------------------------------------------
# 12. Final verdict
# ---------------------------------------------------------------------------
print("\n" + "=" * 100)
print("VERDICT: IS THE ORTHOGONAL LOSS JUST EMBEDDING SHRINKAGE?")
print("=" * 100)

# Compare Frobenius norms
base_drv_frob = norm_stats["model_no_orthogonal"]["drivers"]["frobenius"]
orth_drv_frob = norm_stats["model_orthogonal"]["drivers"]["frobenius"]
drv_shrinkage = 1.0 - orth_drv_frob / base_drv_frob

base_cons_frob = norm_stats["model_no_orthogonal"]["constructors"]["frobenius"]
orth_cons_frob = norm_stats["model_orthogonal"]["constructors"]["frobenius"]
cons_shrinkage = 1.0 - orth_cons_frob / base_cons_frob

base_cos = cosine_df[cosine_df["lambda"] == 0.0]["mean_abs_cos"].values[0]
orth_cos = cosine_df[cosine_df["lambda"] == 1.0]["mean_abs_cos"].values[0]
cos_reduction = 1.0 - orth_cos / (base_cos + 1e-8)

print(f"\nDriver embedding shrinkage:  {drv_shrinkage*100:.1f}%")
print(f"Constructor embedding shrinkage: {cons_shrinkage*100:.1f}%")
print(f"Cosine reduction:             {cos_reduction*100:.1f}%")
print()

SHRINKAGE_THRESHOLD = 0.20  # 20% norm reduction = concerning
COSINE_THRESHOLD = 0.10      # 10% cosine reduction = meaningful

if drv_shrinkage > SHRINKAGE_THRESHOLD or cons_shrinkage > SHRINKAGE_THRESHOLD:
    print("🔴 VERDICT: SHRINKAGE CONFIRMED")
    print(f"   Embedding norms drop >{SHRINKAGE_THRESHOLD*100:.0f}% with orthogonal loss.")
    print("   The loss is being satisfied by shrinking magnitudes, not by learning orthogonality.")
    print("   Recommendation: normalize embeddings before computing orthogonal loss,")
    print("   or add a norm-preservation term (e.g., variance regularization from VICReg).")
elif cos_reduction > COSINE_THRESHOLD and drv_shrinkage < SHRINKAGE_THRESHOLD:
    print("🟢 VERDICT: GENUINE ORTHOGONALITY")
    print(f"   Cosine drops {cos_reduction*100:.1f}% while norms stay within {SHRINKAGE_THRESHOLD*100:.0f}%.")
    print("   The orthogonal loss is producing meaningful decorrelation, not just shrinkage.")
else:
    print("🟡 VERDICT: MIXED OR INCONCLUSIVE")
    print(f"   Shrinkage: drivers={drv_shrinkage*100:.1f}%, constructors={cons_shrinkage*100:.1f}%")
    print(f"   Cosine reduction: {cos_reduction*100:.1f}%")
    print("   The effect is small — may be real but not dramatic.")
    print("   Multi-seed replication needed to confirm.")

## 🔍 Interpretation Notes

### What "shrinkage confirmed" means for your project

If the orthogonal model has significantly smaller norms, you have two paths:

**Path A: Fix the loss.** Normalize embeddings to unit vectors before computing dot product:
```python
z_drv_n = F.normalize(z_drv_paired, p=2, dim=-1)
z_cons_n = F.normalize(z_cons_paired, p=2, dim=-1)
dot_products = torch.sum(z_drv_n * z_cons_n, dim=1)  # now equals cos(θ)
loss = torch.mean(dot_products ** 2)
```
This penalizes ONLY the angle, not the magnitude. Then retrain and compare.

**Path B: Add a variance term** (like VICReg) to prevent collapse:
```python
loss = ortho_loss + gamma * (1.0 / var(drv_emb) + 1.0 / var(cons_emb))
```

### What "genuine orthogonality" means

If norms are comparable but cosine drops, the orthogonal loss is doing its job.
This strengthens your thesis claim significantly — you can say:

> "The improvement is not due to embedding shrinkage (confirmed by Frobenius norm analysis).
> The orthogonal loss induces genuinely decorrelated latent subspaces."

This is the claim that would survive peer review.